### Regression task to learn better 
This notebook combines two implementations
- kfold on tf-idf vectorization + ridge model
- kfold on roberta-base 

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error

In [2]:
DIR = "/kaggle/input/competitions/commonlitreadabilityprize"

train_df = pd.read_csv(f"{DIR}/train.csv")
test_df = pd.read_csv(f"{DIR}/test.csv")

print(train_df.head())

          id url_legal license  \
0  c12129c31       NaN     NaN   
1  85aa80a4c       NaN     NaN   
2  b69ac6792       NaN     NaN   
3  dd1000b26       NaN     NaN   
4  37c1b32fb       NaN     NaN   

                                             excerpt    target  standard_error  
0  When the young people returned to the ballroom... -0.340259        0.464009  
1  All through dinner time, Mrs. Fayre was somewh... -0.315372        0.480805  
2  As Roger had predicted, the snow departed as q... -0.580118        0.476676  
3  And outside before the palace a great garden w... -1.054013        0.450007  
4  Once upon a time there were Three Bears who li...  0.247197        0.510845  


In [3]:
kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
oof_preds = np.zeros(len(train_df)) # array for predictions init with 0s
test_preds = np.zeros(len(test_df))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
    X_tr, y_tr = train_df.iloc[tr_idx]["excerpt"], train_df.iloc[tr_idx]["target"]
    X_val = train_df.iloc[val_idx]["excerpt"]

    model = make_pipeline(
        TfidfVectorizer(ngram_range = (1, 2), max_features = 20000, sublinear_tf = True),
        Ridge(alpha = 1.0)
    )

    model.fit(X_tr, y_tr)

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test_df["excerpt"]) / 5.0

In [4]:
cv_rmse_score = root_mean_squared_error(train_df["target"], oof_preds)
print(cv_rmse_score)

0.7190150309924349


In [32]:
submission = pd.DataFrame({
    "id" : test_df["id"],
    "target" : test_preds
})

submission.to_csv("submission.csv", index = False)
print(submission.head())

          id    target
0  c0f722661 -1.436049
1  f0953f0a5 -0.612656
2  0df072751 -0.476170
3  04caf4e0c -1.655367
4  0e63f8bea -1.259818


In [5]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 256


In [6]:
from transformers import(
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_df = train_df.rename(columns = {"target" : "label"})
train_df["label"] = train_df["label"].astype("float32")


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [8]:
from datasets import Dataset

def tokenize_fn(batch):
    return tokenizer(batch["excerpt"], truncation = True, max_length = MAX_LENGTH)

test_ds = Dataset.from_pandas(test_df[["excerpt"]]).map(
    tokenize_fn, 
    batched = True,
    remove_columns = ["excerpt"]
)

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

In [9]:
from transformers import DataCollatorWithPadding

data_colator = DataCollatorWithPadding(tokenizer = tokenizer)

In [10]:
!pip install -q evaluate

In [14]:
import evaluate

accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = 1)

    rmse = root_mean_squared_error(labels, logits.squeeze())

    accuracy = accuracy_metric.compute(
        predictions = predictions,
        references = labels
    )
    
    return {
        "rmse" : rmse,
        "accuracy" : accuracy["accuracy"]
    }



In [15]:
# remake the kfold

kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
oof_preds = np.zeros(len(train_df)) # array for predictions init with 0s
test_preds = np.zeros(len(test_df))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):

    train_ds = Dataset.from_pandas(
        train_df.iloc[tr_idx][["excerpt", "label"]].reset_index(drop = True)
    ).map(
        tokenize_fn,
        batched = True,
        remove_columns = ["excerpt"]
    )

    val_ds = Dataset.from_pandas(
        train_df.iloc[val_idx][["excerpt", "label"]].reset_index(drop = True)
    ).map(
        tokenize_fn,
        batched = True,
        remove_columns = ["excerpt"]
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels = 1
    )

    args = TrainingArguments(
        output_dir="/kaggle/working/commonlitreadability_model",

        # base hyperparams
        num_train_epochs=3,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        fp16 = True,
        weight_decay = 0.01,
    
        # evaluation and saving
        eval_strategy = "epoch",
        save_strategy = "epoch",
        save_total_limit = 1, # only save one checkpoint
        load_best_model_at_end = True, # gives us the best checkpoint of the model
        metric_for_best_model = "rmse",
    
        # logging
        logging_steps = 50, # log every 50 steps
        report_to = "none"
    )

    trainer = Trainer(
        model = model,
        args = args,
        train_dataset = train_ds,
        eval_dataset = val_ds,
        data_collator = data_colator,
        compute_metrics = compute_metrics
    )

    trainer.train()

    oof_preds[val_idx] = trainer.predict(val_ds).predictions.squeeze()
    test_preds += trainer.predict(test_ds).predictions.squeeze() / 5.0




Map:   0%|          | 0/2267 [00:00<?, ? examples/s]

Map:   0%|          | 0/567 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vect

Epoch,Training Loss,Validation Loss,Rmse,Accuracy
1,1.821631,0.823999,0.641938,0.537919
2,0.811569,0.826572,0.643120,0.537919
3,0.488010,0.694331,0.589433,0.537919


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (45 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [17]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "target": test_preds
})

submission.to_csv("submission.csv", index = False)
print(submission.head(1))

          id    target
0  c0f722661 -0.371006


In [ ]:
# merge the two solutions
oof_blend = 0.90 * oof_preds + 0.10 * oof_preds_ridge
test_blend = 0.90 * test_preds + 0.10 * test_preds_ridge
